# 1. Business Problem & Objective

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 2. Data Loading

In [ ]:
riders = pd.read_csv(r"C:\Users\HP\Downloads\ridewise-churn\data\raw\riders.csv")
trips = pd.read_csv(r"C:\Users\HP\Downloads\ridewise-churn\data\raw\trips.csv")
promotions = pd.read_csv(r"C:\Users\HP\Downloads\ridewise-churn\data\raw\promotions.csv")
sessions = pd.read_csv(r"C:\Users\HP\Downloads\ridewise-churn\data\raw\sessions.csv")
drivers = pd.read_csv(r"C:\Users\HP\Downloads\ridewise-churn\data\raw\drivers.csv")

print("riders:", riders.shape)
print("trips:", trips.shape)
print("promotions:", promotions.shape)
print("sessions:", sessions.shape)
print("drivers:", drivers.shape)

# 3. Data Understanding (Info + Columns)

In [ ]:
riders.head()

In [ ]:
riders.info()

In [ ]:
riders.columns

In [ ]:
trips.head()

In [ ]:
trips.info()

In [ ]:
trips.columns

In [ ]:
promotions.head()

In [ ]:
promotions.info()

In [ ]:
promotions.columns

In [ ]:
sessions.head()

In [ ]:
sessions.info()

In [ ]:
sessions.columns

In [ ]:
drivers.head()

In [ ]:
drivers.info()

In [ ]:
drivers.columns

# 4. Data Cleaning & Datetime Parsing

In [ ]:
sessions = sessions.rename(columns={'rider_id': 'user_id'})

In [ ]:
sessions.columns

In [ ]:
riders['signup_date'] = pd.to_datetime(riders['signup_date'])

In [ ]:
# trips['pickup_time'] = pd.to_datetime(trips['pickup_time'])

In [ ]:
sessions['session_time'] = pd.to_datetime(sessions['session_time'])

In [ ]:
trips['pickup_time'].min(), trips['pickup_time'].max()

In [ ]:
sessions['session_time'].min(), sessions['session_time'].max()

# 5. Churn Definition (30-day inactivity)

In [ ]:
reference_date = trips['pickup_time'].max()
reference_date

In [ ]:
# Convert to datetime first
trips['pickup_time'] = pd.to_datetime(trips['pickup_time'], errors='coerce', utc=True)

# Remove timezone (make it timezone-naive) so subtraction works everywhere
trips['pickup_time'] = trips['pickup_time'].dt.tz_localize(None)

print(trips['pickup_time'].dtype)
trips['pickup_time'].head()

In [ ]:
reference_date = trips['pickup_time'].max()

last_trip = trips.groupby('user_id')['pickup_time'].max().reset_index()

last_trip['days_since_last_trip'] = (reference_date - last_trip['pickup_time']).dt.days

last_trip['churn'] = (last_trip['days_since_last_trip'] > 30).astype(int)

last_trip[['user_id','days_since_last_trip','churn']].head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
last_trip['churn'].value_counts().plot(kind='bar')
plt.title("Churn distribution (1 = churned, 0 = active)")
plt.xlabel("Churn")
plt.ylabel("Number of users")
plt.show()

In [ ]:
plt.figure()
last_trip['days_since_last_trip'].hist(bins=50)
plt.title("Days since last trip distribution")
plt.xlabel("Days")
plt.ylabel("Users")
plt.show()

In [ ]:
# Sanity checks (safe)
print("pickup_time dtype:", trips['pickup_time'].dtype)
print("pickup_time missing:", trips['pickup_time'].isna().sum())
print("pickup_time range:", trips['pickup_time'].min(), "→", trips['pickup_time'].max())

print("\nlast_trip columns:", last_trip.columns.tolist())
print("churn counts:\n", last_trip['churn'].value_counts())
print("\nlast_trip preview:")
last_trip[['user_id','pickup_time','days_since_last_trip','churn']].head()

In [ ]:
last_trip.columns

In [ ]:
last_trip[['days_since_last_trip','churn']].head()